<a href="https://colab.research.google.com/github/Petro490/practic/blob/main/project_iline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psycopg2-binary mimesis tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 56.9 MB/s eta 0:00:00


In [2]:
import datetime
import random
import psycopg2
from psycopg2.extras import execute_values
from mimesis import Person
from mimesis.locales import Locale
from mimesis.enums import Gender
from tabulate import tabulate

# 1. Настройки подключения к вашей базе данных PostgreSQL
DB_SETTINGS = {
    "host": "localhost",
    "port": 5432,
    "database": "project_iline",
    "user": "postgres",
    "password": "suzdalev"  # Укажите свой настоящий пароль!
}

def get_connection():
    return psycopg2.connect(**DB_SETTINGS)

# =====================================================================
# РАЗДЕЛ 1: ГЕНЕРАЦИЯ ДАННЫХ ЧЕРЕЗ MIMESIS (50 000+ СОТРУДНИКОВ)
# =====================================================================
def auto_generate_db():
    print("\n⏳ Старт генерации 50 000+ сотрудников через Mimesis...")
    person = Person(Locale.RU) # Генерирует валидные русские имена

    employees = []
    start_date = datetime.date(2015, 1, 1)

    # Строго задаем мужской пол для соответствия отчеств и фамилий
    g = Gender.MALE

    # Уровень 1: CEO (1 человек)
    employees.append((1, "Иванов Петр Сергеевич", "CEO", datetime.date(2015, 1, 10), 800000.00, None, 1))

    # Уровень 2: Managers (50 человек)
    manager_ids = list(range(2, 52))
    for emp_id in manager_ids:
        name = f"{person.last_name(gender=g)} {person.first_name(gender=g)} {person.surname(gender=g)}"
        salary = round(random.uniform(350000, 500000), 2)
        hire_date = start_date + datetime.timedelta(days=random.randint(100, 1000))
        employees.append((emp_id, name, "Department Manager", hire_date, salary, 1, 2))

    # Уровень 3: Team Leads (1 000 человек)
    tl_ids = list(range(52, 1052))
    for emp_id in tl_ids:
        name = f"{person.last_name(gender=g)} {person.first_name(gender=g)} {person.surname(gender=g)}"
        salary = round(random.uniform(200000, 280000), 2)
        hire_date = start_date + datetime.timedelta(days=random.randint(1000, 2000))
        employees.append((emp_id, name, "Team Lead", hire_date, salary, random.choice(manager_ids), 3))

    # Уровень 4: Senior Developers (9 000 человек)
    senior_ids = list(range(1052, 10052))
    for emp_id in senior_ids:
        name = f"{person.last_name(gender=g)} {person.first_name(gender=g)} {person.surname(gender=g)}"
        salary = round(random.uniform(140000, 190000), 2)
        hire_date = start_date + datetime.timedelta(days=random.randint(2000, 3000))
        employees.append((emp_id, name, "Senior Developer", hire_date, salary, random.choice(tl_ids), 4))

    # Уровень 5: Developers (40 000 человек)
    dev_ids = list(range(10052, 50052))
    for emp_id in dev_ids:
        name = f"{person.last_name(gender=g)} {person.first_name(gender=g)} {person.surname(gender=g)}"
        salary = round(random.uniform(70000, 120000), 2)
        hire_date = start_date + datetime.timedelta(days=random.randint(3000, 4200))
        employees.append((emp_id, name, "Developer", hire_date, salary, random.choice(senior_ids), 5))

    # Запись в PostgreSQL
    try:
        conn = get_connection()
        cursor = conn.cursor()
        cursor.execute("DROP TABLE IF EXISTS employees CASCADE;")
        cursor.execute("""
            CREATE TABLE employees (
                employee_id INT PRIMARY KEY,
                full_name VARCHAR(150) NOT NULL,
                position VARCHAR(100) NOT NULL,
                hire_date DATE NOT NULL,
                salary NUMERIC(10, 2) NOT NULL,
                manager_id INT REFERENCES employees(employee_id),
                hierarchy_level INT
            );
        """)
        query = "INSERT INTO employees VALUES %s"
        execute_values(cursor, query, employees)
        cursor.execute("CREATE INDEX idx_employees_manager ON employees(manager_id);")
        conn.commit()
        print(f"🎉 Успех! Создана иерархия. В базу загружено {len(employees)} сотрудников.")
    except Exception as e:
        print(f"❌ Ошибка инициализации базы: {e}")
    finally:
        cursor.close()
        conn.close()


# =====================================================================
# РАЗДЕЛ 2: КОНСОЛЬНЫЙ ИНТЕРФЕЙС УПРАВЛЕНИЯ (CRUD)
# =====================================================================
def add_employee():
    print("\n--- Добавление сотрудника ---")
    emp_id = int(input("ID сотрудника: "))
    name = input("ФИО: ")
    pos = input("Должность: ")
    date_str = input("Дата приема (ГГГГ-ММ-СС): ")
    salary = float(input("Заработная плата: "))
    m_id = input("ID начальника (оставьте пустым, если нет): ")
    m_id = int(m_id) if m_id else None
    lvl = int(input("Уровень иерархии (1-5): "))

    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO employees VALUES (%s, %s, %s, %s, %s, %s, %s);", (emp_id, name, pos, date_str, salary, m_id, lvl))
        conn.commit()
        print("✅ Сотрудник успешно добавлен!")
    except Exception as e:
        print(f"❌ Ошибка: {e}")
    finally:
        conn.close()

def view_employees():
    print("\n--- Просмотр и фильтрация ---")
    print("1. Показать топ-20 сотрудников")
    print("2. Фильтр по должности")
    print("3. Сортировка по зарплате (по убыванию)")
    choice = input("Выберите подпункт: ").strip()

    conn = get_connection()
    cursor = conn.cursor()

    if choice == "1":
        cursor.execute("SELECT employee_id, full_name, position, salary, manager_id FROM employees LIMIT 20;")
    elif choice == "2":
        pos = input("Введите должность (например, Team Lead): ").strip()
        # Добавляем % с обеих сторон, чтобы искать по части слова (например, 'developer' найдет 'Senior Developer')
        search_param = f"%{pos}%"
        cursor.execute("""
            SELECT employee_id, full_name, position, salary, manager_id
            FROM employees
            WHERE position ILIKE %s
            LIMIT 20;
        """, (search_param,))
    elif choice == "3":
        cursor.execute("SELECT employee_id, full_name, position, salary, manager_id FROM employees ORDER BY salary DESC LIMIT 20;")
    else:
        print("❌ Неверный подпункт!")
        conn.close()
        return

    rows = cursor.fetchall()
    print(tabulate(rows, headers=["ID", "ФИО", "Должность", "Зарплата", "ID Руководителя"], tablefmt="grid"))
    conn.close()

def update_employee():
    print("\n--- Обновление данных ---")
    emp_id = int(input("Введите ID сотрудника для изменения: "))
    new_salary = float(input("Введите новую заработную плату: "))

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("UPDATE employees SET salary = %s WHERE employee_id = %s;", (new_salary, emp_id))
    conn.commit()
    print("✅ Данные обновлены!")
    conn.close()

def delete_employee():
    print("\n--- Удаление сотрудника ---")
    emp_id = int(input("Введите ID сотрудника для удаления: "))

    conn = get_connection()
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM employees WHERE employee_id = %s;", (emp_id,))
        conn.commit()
        print("✅ Сотрудник удален из базы данных.")
    except Exception as e:
        print(f"❌ Не удалось удалить (возможно, он является чьим-то начальником): {e}")
    finally:
        conn.close()

def main_menu():
    while True:
        print("\n===== СУБД ИЛАЙН: МЕНЮ УПРАВЛЕНИЯ ПЕРСОНАЛОМ =====")
        print("1. Автоматическая генерация базы (50 000+ сотрудников через Mimesis)")
        print("2. Добавить нового сотрудника (Create)")
        print("3. Просмотр, фильтрация и сортировка (Read)")
        print("4. Изменить зарплату сотрудника (Update)")
        print("5. Удалить сотрудника из базы (Delete)")
        print("6. Выйти из приложения")

        choice = input("\nВыберите действие (1-6): ")
        if choice == "1": auto_generate_db()
        elif choice == "2": add_employee()
        elif choice == "3": view_employees()
        elif choice == "4": update_employee()
        elif choice == "5": delete_employee()
        elif choice == "6":
            print("Выход из программы. Хорошего дня!")
            break
        else: print("Неверный пункт меню.")

if __name__ == "__main__":
    main_menu()



===== СУБД ИЛАЙН: МЕНЮ УПРАВЛЕНИЯ ПЕРСОНАЛОМ =====
1. Автоматическая генерация базы (50 000+ сотрудников через Mimesis)
2. Добавить нового сотрудника (Create)
3. Просмотр, фильтрация и сортировка (Read)
4. Изменить зарплату сотрудника (Update)
5. Удалить сотрудника из базы (Delete)
6. Выйти из приложения

Выберите действие (1-6): 1

⏳ Старт генерации 50 000+ сотрудников через Mimesis...
❌ Ошибка инициализации базы: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?



UnboundLocalError: cannot access local variable 'cursor' where it is not associated with a value